# Enterprise Document Intelligence System


In [1]:
import sys

!{sys.executable} -m pip install -q langchain
!{sys.executable} -m pip install -q langchain-community
!{sys.executable} -m pip install -q langchain-core
!{sys.executable} -m pip install -q langchain-groq
!{sys.executable} -m pip install -q langchain-huggingface
!{sys.executable} -m pip install -q langchain-chroma
!{sys.executable} -m pip install -q langchain-text-splitters
!{sys.executable} -m pip install -q chromadb
!{sys.executable} -m pip install -q sentence-transformers
!{sys.executable} -m pip install -q pypdf
!{sys.executable} -m pip install -q streamlit
!{sys.executable} -m pip install -q python-dotenv


## Set Your Groq API Key


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [3]:
# Document Loading
from langchain_community.document_loaders import PyPDFLoader

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_chroma import Chroma

# LLM
from langchain_groq import ChatGroq

# Prompts & Chain (Modern LCEL)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All imports loaded successfully ✅")

c:\Aswath\AI-ML\Enterprise_Rag_System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports loaded successfully ✅


In [4]:
import os
from pathlib import Path

# Use relative path (production best practice)
BASE_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
PDF_PATH = BASE_DIR / "Data" / "company_hr_policy.pdf"

# Validate
if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found at: {PDF_PATH}")

print(f"PDF loaded ✅ | Size: {PDF_PATH.stat().st_size:,} bytes")

PDF loaded ✅ | Size: 9,875 bytes


# Load Environment Variables

In [5]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env file
load_dotenv(BASE_DIR / ".env")

# Validate key exists
if not os.environ.get("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in .env file!")

print("Environment loaded ✅")

Environment loaded ✅


# Document Loading & Chunking

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
documents = PyPDFLoader(str(PDF_PATH)).load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = splitter.split_documents(documents)

print(f"Pages: {len(documents)} | Chunks: {len(chunks)} ✅")

Pages: 5 | Chunks: 13 ✅


# Embeddings & Vector Store

In [7]:
import shutil
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = BASE_DIR / "chroma_db"

# Clear old ChromaDB if exists
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Store chunks in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR)
)

print(f"Vectors stored: {vectorstore._collection.count()} ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6560.57it/s]


Vectors stored: 13 ✅


## Build RAG Chain

In [8]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Initialize LLM
llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)

# Anti-hallucination prompt
prompt = PromptTemplate(
    template="""You are a helpful document assistant.
Use ONLY the context below to answer the question.
If the answer is not in the context, say: I do not have enough information in the documents.
Do NOT make up any information.

Context: {context}
Question: {question}
Answer:""",
    input_variables=["context", "question"]
)

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Modern LCEL Chain with source documents
rag_chain = RunnableParallel(
    answer=(
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    ),
    source_documents=(retriever)
)

print("RAG Chain ready ✅")

RAG Chain ready ✅


# Query Function

In [9]:
import os

def ask(question: str) -> dict:
    """Query the RAG chain and return answer with sources."""
    
    result = rag_chain.invoke(question)
    
    # Extract unique sources
    sources = []
    seen = set()
    for doc in result.get("source_documents", []):
        page = int(doc.metadata.get("page", 0)) + 1
        src  = os.path.basename(doc.metadata.get("source", "doc"))
        key  = f"{src}_{page}"
        if key not in seen:
            sources.append({"file": src, "page": page})
            seen.add(key)

    # Confidence score
    score = min(len(seen) * 25, 100)
    confidence = "High" if score >= 75 else "Medium" if score >= 50 else "Low"

    return {
        "question": question,
        "answer": result["answer"],
        "sources": sources,
        "confidence": f"{confidence} ({score}%)"
    }


def display(result: dict):
    """Display result cleanly."""
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Confidence: {result['confidence']}")
    print(f"Sources: {result['sources']}\n")


# Test Queries

In [10]:
questions = [
    "How many days of annual leave do employees get?",
    "What is the work from home policy?",
    "What is the notice period for resignation?",
    "How much is the health insurance coverage?"
]

for q in questions:
    display(ask(q))

Q: How many days of annual leave do employees get?
A: 20 days of annual leave per calendar year.
Confidence: High (75%)
Sources: [{'file': 'company_hr_policy.pdf', 'page': 1}, {'file': 'company_hr_policy.pdf', 'page': 2}, {'file': 'company_hr_policy.pdf', 'page': 3}]

Q: What is the work from home policy?
A: The work from home policy at TechCorp is a hybrid model that requires employees to work from the office a minimum of 3 days per week (Tuesday, Wednesday, and Thursday are mandatory office days). Monday and Friday may be work from home days subject to manager approval and project requirements. Client-facing roles and certain project phases may require full on-site presence as determined by the project manager. Employees approved for regular work from home receive a monthly WFH allowance of Rs. 2,000 to cover internet and electricity expenses. They must ensure a stable internet connection of minimum 25 Mbps and a professional workspace environment for client calls and team meetings.


In [11]:
# Change question below and run!
ask("What is the performance bonus percentage?")

{'question': 'What is the performance bonus percentage?',
 'answer': 'The bonus range is 8% to 25% of annual CTC.',
 'sources': [{'file': 'company_hr_policy.pdf', 'page': 2},
  {'file': 'company_hr_policy.pdf', 'page': 3}],
 'confidence': 'Medium (50%)'}

### Save Production Streamlit App

In [12]:

app_code = '''
import os
import shutil
import streamlit as st
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Load environment
load_dotenv()

# ── Page Config ──────────────────────────────
st.set_page_config(
    page_title="Document Intelligence",
    page_icon="📄",
    layout="wide"
)

st.title("📄 Document Intelligence System")
st.caption("Upload PDFs and ask questions with source citations")

# ── Helpers ──────────────────────────────────
DATA_DIR   = Path("data")
CHROMA_DIR = Path("chroma_db")

def build_chain(files):
    DATA_DIR.mkdir(exist_ok=True)
    for f in files:
        (DATA_DIR / f.name).write_bytes(f.getbuffer())

    if CHROMA_DIR.exists():
        shutil.rmtree(CHROMA_DIR)

    docs = []
    for f in files:
        docs.extend(PyPDFLoader(str(DATA_DIR / f.name)).load())

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    ).split_documents(docs)

    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    vectorstore = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=str(CHROMA_DIR)
    )

    llm = ChatGroq(
        groq_api_key=os.environ["GROQ_API_KEY"],
        model_name="llama-3.3-70b-versatile",
        temperature=0.1,
        max_tokens=1024
    )

    prompt = PromptTemplate(
        template="""Use ONLY the context below to answer.
If not found, say: I do not have enough information.
Do NOT make up any information.

Context: {context}
Question: {question}
Answer:""",
        input_variables=["context", "question"]
    )

    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

    def format_docs(docs):
        return "\\n\\n".join(doc.page_content for doc in docs)

    return RunnableParallel(
        answer=(
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt | llm | StrOutputParser()
        ),
        source_documents=retriever
    )


# ── Sidebar ───────────────────────────────────
with st.sidebar:
    st.header("📂 Upload PDF")
    files = st.file_uploader(
        "Choose PDF files",
        type=["pdf"],
        accept_multiple_files=True
    )

    if st.button("⚡ Index Documents", type="primary", use_container_width=True):
        if not files:
            st.error("Upload a PDF first!")
            st.stop()
        with st.spinner("Processing... please wait..."):
            st.session_state["chain"] = build_chain(files)
        st.success(f"{len(files)} file(s) indexed! Ask questions below ✅")


# ── Chat Interface ────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []

for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

if question := st.chat_input("Ask about your documents..."):
    if "chain" not in st.session_state:
        st.error("Please index documents first!")
        st.stop()

    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.spinner("Searching..."):
            result = st.session_state["chain"].invoke(question)

        st.markdown(result["answer"])

        # Show sources
        seen = set()
        for doc in result.get("source_documents", []):
            page = int(doc.metadata.get("page", 0)) + 1
            src  = Path(doc.metadata.get("source", "doc")).name
            key  = f"{src}_{page}"
            if key not in seen:
                st.caption(f"📄 {src} — Page {page}")
                seen.add(key)

    st.session_state.messages.append({
        "role": "assistant",
        "content": result["answer"]
    })
'''

# Save app
with open("rag_app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip())

print("rag_app.py created ✅")
print("\nTo launch: streamlit run rag_app.py")

rag_app.py created ✅

To launch: streamlit run rag_app.py
